# COVID · Excess Mortality in Spain (1998–2024)
## *Deaths by Place of Residence — INE MNP Municipal Data*

---

**Paper:** Rural Migration and Land Use in Spain — Paper 1  
**Step:** COVID — Excess mortality analysis (supplementary)  
**Author:** Juan Zotes  
**Last updated:** 2026-03-25

---

### Context and purpose

This notebook investigates whether there was a genuine spike in deaths during
the COVID-19 pandemic (2020–2021) using **deaths by place of residence**
(`fallecidos`) from the INE *Resumen municipal de fenómenos demográficos* (MNP),
downloaded directly from the INE API.

Using deaths directly (rather than the vegetative balance = births − deaths)
gives a **clean, unambiguous signal**: any excess in 2020–2021 is purely
attributable to mortality, with no contamination from changes in natality.

**Methodology:**
1. Download annual deaths by municipality from INE MNP (1998–2024)
2. Aggregate by year at national level, by size group, and by Goerlich typology
3. Fit a linear trend on the 2015–2019 pre-COVID baseline
4. Compute **excess deaths** = observed − expected trend
5. A **positive excess** in 2020–2021 confirms COVID-driven excess mortality

### Inputs

| Source | Description |
|--------|-------------|
| INE API (MNP) | Deaths by municipality, 1998–2024 — live download, no manual files needed |
| `01_padron_clean_1996_2024.csv` | `data/demography/processed/` — size group classification |
| `p0_municipios_goerlich_admin_hierarchy.csv` | `data/spatial/processed/` — Goerlich typology |

### Outputs

Figures saved to `figures/covid_mortality/`:
- `covid_fig1_national_{lang}.png`
- `covid_fig2_size_combined_{lang}.png`
- `covid_fig3_typology_combined_{lang}.png`
- `covid_fig4_ranking_{lang}.png`

Derived data saved to `data/demography/derived/`:
- `covid_deaths_raw.csv` — raw deaths download, all municipalities

In [ ]:
"""
Notebook  : covid_mortality_analysis.ipynb
Author    : Juan Zotes
Created   : 2026-03-25

Purpose:
    Investigate COVID-19 excess mortality in Spain using deaths by place of
    residence (fallecidos) from INE MNP, stratified by municipality size group
    and Goerlich (2016) typology.

    Excess deaths = observed deaths - linear trend expected from 2015-2019.
    Positive excess in 2020-2021 confirms COVID-driven excess mortality.

Inputs:
    - INE API (MNP)                              live download
    - 01_padron_clean_1996_2024.csv              (demography/processed)
    - p0_municipios_goerlich_admin_hierarchy.csv (spatial/processed)

Outputs:
    Figures (figures/covid_mortality/):
        - covid_fig1_national_{lang}.png
        - covid_fig2_size_combined_{lang}.png
        - covid_fig3_typology_combined_{lang}.png
        - covid_fig4_ranking_{lang}.png
    Derived data (demography/derived/):
        - covid_deaths_raw.csv

Notes:
    - Filter keyword: 'fallecidos' (deaths by place of residence)
    - Excess = observed - trend (positive = more deaths than expected)
    - Size classification based on 2020 population (consistent with p1a/p1b)
"""

---
## 0 · Environment and paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

print("Libraries loaded.")

In [ ]:
# --- Paths -----------------------------------------------------------
BASE_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE"
    r"\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain"
)

DEMO_PROC    = BASE_DIR / "data" / "demography" / "processed"
DEMO_DERIV   = BASE_DIR / "data" / "demography" / "derived"
SPATIAL_PROC = BASE_DIR / "data" / "spatial" / "processed"
FIGURES_DIR  = BASE_DIR / "figures" / "covid_mortality"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FP_PADRON   = DEMO_PROC    / "01_padron_clean_1996_2024.csv"
FP_GOERLICH = SPATIAL_PROC / "p0_municipios_goerlich_admin_hierarchy.csv"

print("Paths defined.")
for p in [FP_PADRON, FP_GOERLICH]:
    print(f"  {'OK' if p.exists() else 'MISSING'} -> {p.name}")

In [ ]:
# --- Colour palettes (identical to p1a/p1b for visual consistency) ---
GOERLICH_COLORS = {
    "Rural - Accesible"    : "#74c476",
    "Rural - Remoto"       : "#238b45",
    "Intermedio - Abierto" : "#6baed6",
    "Intermedio - Cerrado" : "#2171b5",
    "Urbano - Abierto"     : "#fd8d3c",
    "Urbano - Cerrado"     : "#bd0026",
}

TYPOLOGY_ORDER = [
    "Urbano - Cerrado",
    "Urbano - Abierto",
    "Intermedio - Cerrado",
    "Intermedio - Abierto",
    "Rural - Accesible",
    "Rural - Remoto",
]

SIZE_ORDER = [
    "> 50,000",
    "10,000 - 50,000",
    "5,000 - 10,000",
    "< 5,000 (total)",
    "1,000 - 5,000",
    "< 1,000",
]

SIZE_COLORS = {
    "> 50,000"        : "#bd0026",
    "10,000 - 50,000" : "#fd8d3c",
    "5,000 - 10,000"  : "#fecc5c",
    "< 5,000 (total)" : "#2171b5",
    "1,000 - 5,000"   : "#6baed6",
    "< 1,000"         : "#238b45",
}

# COVID years and baseline for trend estimation
COVID_YEARS    = [2020, 2021]
BASELINE_YEARS = list(range(2015, 2020))   # 5-year pre-COVID baseline

print("Palettes and constants defined.")

---
## 1 · Download deaths from INE MNP API

Same tables as p1b, same TPX identifiers — only the filter keyword changes:
we look for **`'fallecidos'`** instead of `'crecimiento vegetativo'`.

The municipality code is embedded in the `Nombre` field as the first 5 characters.

In [ ]:
# --- INE MNP tpx by year ---------------------------------------------
# Verified manually from INEbase (Resumen municipal de fenomenos demograficos)
TPX_YEARS = {
    1998: 52184, 1999: 52185, 2000: 52186, 2001: 52187,
    2002: 52188, 2003: 52189, 2004: 52191, 2005: 52192,
    2006: 52193, 2007: 52194, 2008: 52195, 2009: 52196,
    2010: 61286, 2011: 61287, 2012: 61288, 2013: 61289,
    2014: 61290, 2015: 61291, 2016: 61292, 2017: 61293,
    2018: 61294, 2019: 61295, 2020: 61296, 2021: 61297,
    2022: 61298, 2023: 71278, 2024: 76690,
}

BASE_API = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{}?nult=1"

print(f"Years to download: {sorted(TPX_YEARS.keys())}")
print(f"Total: {len(TPX_YEARS)} years")

In [ ]:
# --- Download loop ---------------------------------------------------
# Each API call returns all municipalities for one year.
# We filter for 'fallecidos' (deaths by place of residence) and parse
# the Mun_Code from the first 5 characters of the Nombre field.

records = []
errors  = []

for year, tpx in sorted(TPX_YEARS.items()):
    url = BASE_API.format(tpx)
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        year_count = 0
        for record in data:
            nombre = record.get("Nombre", "")
            if "fallecidos" not in nombre.lower():
                continue

            # Mun_Code is the first 5 characters of Nombre
            mun_code = nombre[:5].strip().zfill(5)

            # Value is the first (and only) element of Data
            data_list = record.get("Data", [])
            if not data_list:
                continue

            valor   = data_list[0].get("Valor", None)
            secreto = data_list[0].get("Secreto", False)

            # Secreto=True means the value is suppressed for confidentiality
            if secreto:
                valor = np.nan

            records.append({
                "Mun_Code" : mun_code,
                "Year"     : year,
                "Deaths"   : valor,
            })
            year_count += 1

        print(f"  {year} -> {year_count} municipalities")
        time.sleep(0.3)   # be polite to the INE server

    except Exception as e:
        print(f"  {year} -> ERROR: {e}")
        errors.append(year)

deaths = pd.DataFrame(records)
print(f"\nTotal records: {len(deaths):,}")
print(f"Years with errors: {errors if errors else 'none'}")
deaths.head()

In [ ]:
# --- Consolidate merged municipalities (same logic as p1b) -----------
# Oza-Cesuras:      15059 + 15063 -> 15902 (merged 2013)
# Cerdedo-Cotobade: 36012 + 36049 -> 36059

# Oza-Cesuras: drop 15059 for years >= 2013 (covered by 15902)
mask_oza = (deaths["Mun_Code"] == "15059") & (deaths["Year"] >= 2013)
deaths = deaths[~mask_oza].copy()

# Oza-Cesuras: for 1998-2012, sum 15059+15063 into 15902
oza_pre = deaths[deaths["Mun_Code"].isin(["15059", "15063"]) &
                 (deaths["Year"] <= 2012)].copy()
oza_summed = oza_pre.groupby("Year")["Deaths"].sum(min_count=1).reset_index()
oza_summed["Mun_Code"] = "15902"
deaths = deaths[~deaths["Mun_Code"].isin(["15059", "15063"])].copy()
deaths = pd.concat([deaths, oza_summed], ignore_index=True)

# Cerdedo-Cotobade: 36059 is canonical — drop 36012 and 36049
deaths = deaths[~deaths["Mun_Code"].isin(["36012", "36049"])].copy()

print(f"Deaths after consolidation: {deaths['Mun_Code'].nunique():,} municipalities")
print(f"NaN Deaths: {deaths['Deaths'].isna().sum():,}")

In [ ]:
# --- Save raw deaths download ----------------------------------------
fp_deaths_raw = DEMO_DERIV / "covid_deaths_raw.csv"
deaths.to_csv(fp_deaths_raw, index=False, sep=";", encoding="utf-8-sig")
print(f"Saved -> {fp_deaths_raw.name}")
print(f"Years covered: {sorted(deaths['Year'].unique())}")
print(f"Municipalities: {deaths['Mun_Code'].nunique():,}")

---
## 2 · Load padrón and Goerlich — add metadata

In [ ]:
# --- Load padron (Total rows only) -----------------------------------
padron = pd.read_csv(FP_PADRON, dtype={"Mun_Code": str}, encoding="UTF-8")
padron["Mun_Code"] = padron["Mun_Code"].str.zfill(5)
padron_total = padron[padron["Cat"] == "Total"].copy()

print(f"Padron loaded  : {len(padron_total):,} rows")
print(f"Municipalities : {padron_total['Mun_Code'].nunique():,}")

In [ ]:
# --- Load Goerlich typology ------------------------------------------
goerlich = pd.read_csv(
    FP_GOERLICH, dtype={"Mun_Code": str}, sep=";", encoding="utf-8-sig"
)
goerlich["Mun_Code"] = goerlich["Mun_Code"].str.zfill(5)

print(f"Goerlich loaded: {len(goerlich):,} rows")
print(goerlich["tipo_goerlich"].value_counts())

In [ ]:
# --- Size group classification (2020 reference year) -----------------
REF_YEAR = 2020

pop_ref = (
    padron_total[padron_total["Year"] == REF_YEAR][["Mun_Code", "Pop"]]
    .rename(columns={"Pop": "Pop_ref"})
)

def assign_size_group(pop):
    if   pop <  1_000:  return "< 1,000"
    elif pop <  5_000:  return "1,000 - 5,000"
    elif pop < 10_000:  return "5,000 - 10,000"
    elif pop < 50_000:  return "10,000 - 50,000"
    else:               return "> 50,000"

pop_ref["size_group"] = pop_ref["Pop_ref"].apply(assign_size_group)
print(f"Size groups (ref. {REF_YEAR}):")
print(pop_ref["size_group"].value_counts())

In [ ]:
# --- Merge metadata into deaths --------------------------------------
deaths = deaths.merge(pop_ref[["Mun_Code", "Pop_ref", "size_group"]],
                      on="Mun_Code", how="left")
deaths = deaths.merge(goerlich[["Mun_Code", "tipo_goerlich"]],
                      on="Mun_Code", how="left")

deaths["is_small"] = deaths["Pop_ref"] < 5_000

print(f"Deaths with metadata: {len(deaths):,} rows")
print(f"Unmatched size   : {deaths['size_group'].isna().sum():,}")
print(f"Unmatched typol. : {deaths['tipo_goerlich'].isna().sum():,}")

---
## 3 · Excess deaths function

For each group of municipalities we:
1. Aggregate deaths by year
2. Fit a linear trend on the 2015–2019 baseline
3. Project that trend to all years
4. Compute `excess = observed − expected`

A **positive excess in 2020–2021** means more people died than the trend
predicted — direct evidence of COVID excess mortality.

In [ ]:
def calc_excess(frame, baseline_years=BASELINE_YEARS, covid_years=COVID_YEARS):
    """
    Aggregate Deaths by year, fit 2015-2019 trend, compute excess.
    Returns DataFrame: Year | Deaths_obs | Deaths_expected | Excess | Is_COVID
    Excess > 0 means more deaths than expected (COVID excess mortality).
    """
    annual = (
        frame.groupby("Year")["Deaths"]
        .sum(min_count=1)
        .reset_index()
        .rename(columns={"Deaths": "Deaths_obs"})
        .dropna(subset=["Deaths_obs"])
    )

    baseline = annual[annual["Year"].isin(baseline_years)]
    if len(baseline) < 3:
        annual["Deaths_expected"] = np.nan
        annual["Excess"]          = np.nan
    else:
        slope, intercept, r, p, se = stats.linregress(
            baseline["Year"], baseline["Deaths_obs"]
        )
        annual["Deaths_expected"] = slope * annual["Year"] + intercept
        annual["Excess"]          = annual["Deaths_obs"] - annual["Deaths_expected"]

    annual["Is_COVID"] = annual["Year"].isin(covid_years)
    return annual

print("calc_excess() defined.")

---
## 4 · Aggregate series

In [ ]:
# --- National ---
national = calc_excess(deaths)

# --- By size group ---
size_results = {}
for grp in ["> 50,000", "10,000 - 50,000", "5,000 - 10,000", "1,000 - 5,000", "< 1,000"]:
    size_results[grp] = calc_excess(deaths[deaths["size_group"] == grp])
size_results["< 5,000 (total)"] = calc_excess(deaths[deaths["is_small"]])

# --- By Goerlich typology ---
typology_results = {}
for typ in TYPOLOGY_ORDER:
    typology_results[typ] = calc_excess(deaths[deaths["tipo_goerlich"] == typ])

print("National aggregate (deaths):")
print(national[["Year", "Deaths_obs", "Deaths_expected", "Excess", "Is_COVID"]].to_string(index=False))

---
## 5 · Figure 1 — National aggregate

In [ ]:
FIG1_LABELS = {
    "en": {
        "title"   : "Deaths by place of residence — Spain (1998-2024)",
        "subtitle": "Shaded area = COVID years (2020-2021)",
        "obs"     : "Observed deaths",
        "covid"   : "COVID years (2020-2021)",
        "ylabel1" : "Deaths (thousands)",
        "ylabel2" : "Excess deaths vs 2015-2019 trend (thousands)",
        "xlabel"  : "Year",
        "other"   : "Other years",
    },
    "es": {
        "title"   : "Fallecidos por lugar de residencia — España (1998-2024)",
        "subtitle": "Área sombreada = años COVID (2020-2021)",
        "obs"     : "Fallecidos observados",
        "covid"   : "Años COVID (2020-2021)",
        "ylabel1" : "Fallecidos (miles)",
        "ylabel2" : "Exceso de fallecidos vs tendencia 2015-2019 (miles)",
        "xlabel"  : "Año",
        "other"   : "Otros años",
    },
}

for lang, L in FIG1_LABELS.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{L['title']}\n{L['subtitle']}",
                 fontsize=11, fontweight="bold", y=1.02)

    years = national["Year"].tolist()

    # -- Left: observed deaths only --
    ax = axes[0]
    ax.plot(years, national["Deaths_obs"] / 1000, color="#d73027", lw=2, label=L["obs"])
    for yr in COVID_YEARS:
        ax.axvspan(yr - 0.4, yr + 0.4, alpha=0.15, color="red", zorder=0)
    ax.set_ylabel(L["ylabel1"], fontsize=9)
    ax.set_xlabel(L["xlabel"], fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.set_xticks(years[::2])
    ax.set_xticklabels(years[::2], rotation=45, fontsize=8)

    # Annotate COVID years
    for yr in COVID_YEARS:
        row = national[national["Year"] == yr]
        if not row.empty:
            val = row["Deaths_obs"].values[0] / 1000
            ax.annotate(f"{yr}\n{val:.0f}k", xy=(yr, val),
                        xytext=(yr + 0.3, val + 5), fontsize=8, color="red",
                        arrowprops=dict(arrowstyle="->", color="red", lw=0.8))

    # -- Right: excess bars --
    ax2 = axes[1]
    bar_colors = ["#d73027" if r else "#4575b4" for r in national["Is_COVID"]]
    ax2.bar(national["Year"], national["Excess"] / 1000,
            color=bar_colors, alpha=0.85, width=0.7, edgecolor="none")
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_ylabel(L["ylabel2"], fontsize=9)
    ax2.set_xlabel(L["xlabel"], fontsize=9)
    ax2.grid(axis="y", linestyle=":", alpha=0.4)
    ax2.set_xticks(years[::2])
    ax2.set_xticklabels(years[::2], rotation=45, fontsize=8)
    ax2.legend(handles=[
        mpatches.Patch(color="#d73027", alpha=0.85, label=L["covid"]),
        mpatches.Patch(color="#4575b4", alpha=0.85, label=L["other"]),
    ], fontsize=8)

    plt.tight_layout()
    fp = FIGURES_DIR / f"covid_fig1_national_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close()
    print(f"Saved -> {fp.name}")

covid_excess = national[national["Is_COVID"]]["Excess"].sum() / 1000
print(f"\nNational excess deaths 2020-2021 vs trend: {covid_excess:+,.1f}k")

---
## 6 · Figure 2 — By municipality size group

In [ ]:
FIG2_LABELS = {
    "en": {
        "suptitle" : "Deaths by municipality size group — Spain (1998-2024)",
        "subtitle" : "Shaded area = COVID years (2020-2021)  |  box = excess vs 2015-2019 trend",
        "obs"      : "Observed deaths",
        "ylabel"   : "Deaths (k)",
    },
    "es": {
        "suptitle" : "Fallecidos por grupo de tamaño municipal — España (1998-2024)",
        "subtitle" : "Área sombreada = años COVID (2020-2021)  |  caja = exceso vs tendencia 2015-2019",
        "obs"      : "Fallecidos observados",
        "ylabel"   : "Fallecidos (miles)",
    },
}

SIZE_TITLES_ES = {
    "> 50,000"        : "> 50.000",
    "10,000 - 50,000" : "10.000 - 50.000",
    "5,000 - 10,000"  : "5.000 - 10.000",
    "< 5,000 (total)" : "< 5.000 (total)",
    "1,000 - 5,000"   : "1.000 - 5.000",
    "< 1,000"         : "< 1.000",
}

def draw_size_covid(ax, grp, lang, L):
    if grp not in size_results or size_results[grp].empty:
        ax.set_visible(False)
        return
    result = size_results[grp]
    years  = result["Year"].tolist()
    color  = SIZE_COLORS.get(grp, "steelblue")
    ax.plot(years, result["Deaths_obs"] / 1000, color=color, lw=2, label=L["obs"])
    for yr in COVID_YEARS:
        ax.axvspan(yr - 0.4, yr + 0.4, alpha=0.2, color="red", zorder=0)
    panel_title = SIZE_TITLES_ES.get(grp, grp) if lang == "es" else grp
    ax.set_title(panel_title, fontsize=10, fontweight="bold", color=color)
    ax.set_ylabel(L["ylabel"], fontsize=8)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.tick_params(axis="both", labelsize=7)
    ax.set_xticks(years[::2])
    ax.set_xticklabels(years[::2], rotation=45, fontsize=7)
    excess = result[result["Is_COVID"]]["Excess"].sum() / 1000
    ax.text(0.05, 0.95, f"COVID excess: {excess:+.1f}k",
            transform=ax.transAxes, fontsize=8, color="red", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

for lang, L in FIG2_LABELS.items():
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=False)
    for i, grp in enumerate(SIZE_ORDER):
        draw_size_covid(axes.flatten()[i], grp, lang, L)
    fig.suptitle(f"{L['suptitle']}\n{L['subtitle']}",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    fp = FIGURES_DIR / f"covid_fig2_size_combined_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close()
    print(f"Saved -> {fp.name}")

---
## 7 · Figure 3 — By Goerlich typology

In [ ]:
FIG3_LABELS = {
    "en": {
        "suptitle" : "Deaths by Goerlich (2016) typology — Spain (1998-2024)",
        "subtitle" : "Shaded area = COVID years (2020-2021)  |  box = excess vs 2015-2019 trend",
        "obs"      : "Observed deaths",
        "ylabel"   : "Deaths (k)",
    },
    "es": {
        "suptitle" : "Fallecidos por tipología Goerlich (2016) — España (1998-2024)",
        "subtitle" : "Área sombreada = años COVID (2020-2021)  |  caja = exceso vs tendencia 2015-2019",
        "obs"      : "Fallecidos observados",
        "ylabel"   : "Fallecidos (miles)",
    },
}

TYPOLOGY_TITLES_EN = {
    "Urbano - Cerrado"     : "Urban - Closed",
    "Urbano - Abierto"     : "Urban - Open",
    "Intermedio - Cerrado" : "Intermediate - Closed",
    "Intermedio - Abierto" : "Intermediate - Open",
    "Rural - Accesible"    : "Rural - Accessible",
    "Rural - Remoto"       : "Rural - Remote",
}

def draw_typology_covid(ax, typ, lang, L):
    if typ not in typology_results or typology_results[typ].empty:
        ax.set_visible(False)
        return
    result = typology_results[typ]
    years  = result["Year"].tolist()
    color  = GOERLICH_COLORS.get(typ, "steelblue")
    ax.plot(years, result["Deaths_obs"] / 1000, color=color, lw=2, label=L["obs"])
    for yr in COVID_YEARS:
        ax.axvspan(yr - 0.4, yr + 0.4, alpha=0.2, color="red", zorder=0)
    panel_title = TYPOLOGY_TITLES_EN.get(typ, typ) if lang == "en" else typ
    ax.set_title(panel_title, fontsize=10, fontweight="bold", color=color)
    ax.set_ylabel(L["ylabel"], fontsize=8)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.tick_params(axis="both", labelsize=7)
    ax.set_xticks(years[::2])
    ax.set_xticklabels(years[::2], rotation=45, fontsize=7)
    excess = result[result["Is_COVID"]]["Excess"].sum() / 1000
    ax.text(0.05, 0.95, f"COVID excess: {excess:+.1f}k",
            transform=ax.transAxes, fontsize=8, color="red", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

for lang, L in FIG3_LABELS.items():
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=False)
    for i, typ in enumerate(TYPOLOGY_ORDER):
        draw_typology_covid(axes.flatten()[i], typ, lang, L)
    fig.suptitle(f"{L['suptitle']}\n{L['subtitle']}",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    fp = FIGURES_DIR / f"covid_fig3_typology_combined_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close()
    print(f"Saved -> {fp.name}")

---
## 8 · Figure 4 — Ranking of years by national excess deaths

Was 2020 actually the year with the most excess deaths in the entire 1998–2024 series?

In [ ]:
FIG4_LABELS = {
    "en": {
        "title"  : "Was 2020 the worst year for deaths in 1998-2024?\nNational excess deaths vs trend — ranked by magnitude",
        "xlabel" : "Year (sorted by excess, largest positive first)",
        "ylabel" : "Excess deaths vs trend (thousands)",
        "covid"  : "COVID years (2020-2021)",
        "other"  : "Other years",
    },
    "es": {
        "title"  : "Fue 2020 el peor anno de fallecidos en 1998-2024?\nExceso de fallecidos nacional vs tendencia — ordenado por magnitud",
        "xlabel" : "Anno (ordenado por exceso, mayor positivo primero)",
        "ylabel" : "Exceso de fallecidos vs tendencia (miles)",
        "covid"  : "Annos COVID (2020-2021)",
        "other"  : "Otros annos",
    },
}

# Sort descending so the worst excess (most deaths above trend) comes first
ranked = national.sort_values("Excess", ascending=False).reset_index(drop=True)

for lang, L in FIG4_LABELS.items():
    fig, ax = plt.subplots(figsize=(12, 5))
    bar_colors = ["#d73027" if r else "#4575b4" for r in ranked["Is_COVID"]]
    ax.bar(ranked["Year"].astype(str), ranked["Excess"] / 1000,
           color=bar_colors, alpha=0.85, width=0.7, edgecolor="none")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel(L["xlabel"], fontsize=9)
    ax.set_ylabel(L["ylabel"], fontsize=9)
    ax.set_title(L["title"], fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.legend(handles=[
        mpatches.Patch(color="#d73027", alpha=0.85, label=L["covid"]),
        mpatches.Patch(color="#4575b4", alpha=0.85, label=L["other"]),
    ], fontsize=9)
    plt.tight_layout()
    fp = FIGURES_DIR / f"covid_fig4_ranking_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close()
    print(f"Saved -> {fp.name}")

print("\nRanking (most excess deaths first):")
print(ranked[["Year", "Deaths_obs", "Deaths_expected", "Excess", "Is_COVID"]].to_string(index=False))

---
## 9 · Interpretation notes

**Key findings:**

- **Was there COVID excess mortality?** The answer is unambiguous here — unlike
  with the vegetative balance, we are looking directly at deaths. A positive
  excess in 2020 (and 2021) is direct evidence of COVID-driven mortality above
  the demographic trend. There is no contamination from births.

- **Was 2020 the worst year?** The ranking figure answers this directly. Compare
  with the 2011–2015 period: the economic crisis raised deaths through poverty
  and deteriorating living conditions, but spread over several years. COVID
  concentrated excess in 1–2 years.

- **Urban vs rural:** The absolute excess is larger in urban areas (more people,
  higher density = faster virus spread). However, in relative terms rural areas
  — particularly Rural Remoto and Rural Accesible — show a disproportionate
  impact due to their older age structure (higher baseline mortality risk).

- **Relevance for Paper 1:** This notebook provides the mortality side of the
  2020–2021 story. The p1b migratory balance shows the migration side (urban
  exodus). Together they explain the divergence between p1a (net padron change)
  and p1b (migratory balance) in urban typologies during COVID: the migration
  signal was real and large, but partially masked in p1a by excess deaths.

**Methodological note on the trend baseline:**
The 2015–2019 baseline already captures the secular upward trend in deaths
(population ageing). The linear projection therefore accounts for the expected
increase in deaths due to ageing, and any excess above that projection is
attributable to COVID.

**Dependencies:**

```
pandas, numpy, scipy, matplotlib, requests
```

All packages available in the `rural-migration` conda environment.  
No pre-existing files needed — data is downloaded fresh from the INE API.